In [1]:
import os
import pandas as pd

# Odczyt datasetu

In [2]:
df = pd.read_csv(os.path.join("../data", "domy.csv"))

# Usuwanie kolumn

# Zamiana kilku danych tekstowych na numeryczne

In [3]:
del_cols = ["Id", "Street", "Utilities", "Condition2", "PoolQC", "PoolArea", "LowQualFinSF", "3SsnPorch"]
df = df.copy().drop(del_cols, axis=1)

In [4]:
df = df.copy()
df["LotFrontage"] = pd.to_numeric(df["LotFrontage"], errors="coerce")
df["MasVnrArea"] = pd.to_numeric(df["MasVnrArea"], errors="coerce")
df["GarageYrBlt"] = pd.to_numeric(df["GarageYrBlt"], errors="coerce")

# Usuwanie outlierów

In [5]:
out_cols = ["LotArea", "GrLivArea", "Fireplaces", "HalfBath", "GarageCars"]
ranges = [100000, 5000, 2, 1, 3]

df = df.copy()
for i in range(len(out_cols)):
    df = df[df[out_cols[i]] <= ranges[i]]

# Podział danych na zbiór treningowy i testowy

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inicjalizacja modelu i reportera do treningów

In [7]:
from Regression.training_reporter import TrainingReporter
from Regression.training_model import CustomXGBRegressorModel

model = CustomXGBRegressorModel()
reporter = TrainingReporter(model, X_train, X_test, y_train, y_test)

# Pierwszy trening

In [8]:
reporter.train()

Start training...
Training finished!
Train RMSE: 5598.074490393996
---------------------------------------------------


# Kroswalidacja

In [9]:
reporter.run_cross_validation(cv=10)

Start cross validation...
Fold 0: RMSE 27260.017314741384
Fold 1: RMSE 26508.73969090194
Fold 2: RMSE 24189.44298655924
Fold 3: RMSE 21941.062873069754
Fold 4: RMSE 20727.837899790706
Fold 5: RMSE 50314.56377630636
Fold 6: RMSE 26592.34777149246
Fold 7: RMSE 33015.1616079643
Fold 8: RMSE 28460.154883626336
Fold 9: RMSE 22261.611801484636
RMSE for cross validation: 28127.09406059371 +- 8154.380645181701
Cross validation finished!
---------------------------------------------------


# Grid search

In [10]:
grid_params = reporter.run_grid_search(cv=5)

Start grid search...
Grid finished!
Best params: {'model__colsample_bytree': 0.6, 'model__learning_rate': 0.01, 'model__max_depth': 4, 'model__min_child_weight': 2, 'model__n_estimators': 2000, 'model__subsample': 0.8}
Best RMSE score: -28506.540625
---------------------------------------------------


# Kroswalidacja po dostosowaniu hiperparametrów za pomocą Grid Search

In [12]:
from Regression.training_model import build_model_from_grid_params

model_GS = build_model_from_grid_params(grid_params.best_params_)
reporter_GS = TrainingReporter(model_GS, X_train, X_test, y_train, y_test)
reporter_GS.run_cross_validation(cv=10)

Start cross validation...
Fold 0: RMSE 26001.762401806536
Fold 1: RMSE 27388.923892697938
Fold 2: RMSE 21838.503245414966
Fold 3: RMSE 22875.714983361722
Fold 4: RMSE 20064.58212871626
Fold 5: RMSE 50045.10989097736
Fold 6: RMSE 26494.434736374355
Fold 7: RMSE 37022.27999461946
Fold 8: RMSE 27344.78875398382
Fold 9: RMSE 21411.318128503906
RMSE for cross validation: 28048.741815645633 +- 8631.51765098422
Cross validation finished!
---------------------------------------------------


# Randomized grid search

In [13]:
random_grid = reporter.run_randomized_search(cv=5)

Start randomized grid search...
Randomized search finished!
Best params: {'model__subsample': 0.5, 'model__reg_lambda': 1.0, 'model__reg_alpha': 5.0, 'model__n_estimators': np.int64(1100), 'model__min_child_weight': np.int64(4), 'model__max_depth': np.int64(3), 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.5}
Best RMSE score: -28940.486328125
---------------------------------------------------


# Kroswalidacja po dostosowaniu hiperparametrów za pomocą Grid Search

In [14]:
model_RGS = build_model_from_grid_params(random_grid.best_params_)
reporter_RGS = TrainingReporter(model_RGS, X_train, X_test, y_train, y_test)
reporter_RGS.run_cross_validation(cv=10)

Start cross validation...
Fold 0: RMSE 26694.352960879198
Fold 1: RMSE 27490.109348636648
Fold 2: RMSE 21519.461703304754
Fold 3: RMSE 22877.785207488945
Fold 4: RMSE 19477.74607083684
Fold 5: RMSE 52283.43676538489
Fold 6: RMSE 26425.577912318207
Fold 7: RMSE 32350.422315636006
Fold 8: RMSE 26287.545948604635
Fold 9: RMSE 21441.593970598362
RMSE for cross validation: 27684.803220368853 +- 8937.95253117036
Cross validation finished!
---------------------------------------------------
